# M04B · Matplotlib: The Foundation of Biological Plotting
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2026-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M04_Visualisation/M04B_Matplotlib.ipynb)

⏱ ~120 min · Difficulty: ★★☆ · Prerequisites: M01A (Core Python), M04A (Biological Plots Theory)


## 1. Learning Outcomes

By the end of this notebook, you will be able to:

- **Differentiate**: Contrast Matplotlib's stateful pyplot interface with the object-oriented (Figure, Axes) paradigm and apply OO plotting consistently.
- **Construct**: Create the six standard biological plot types: Bar, Histogram, Pie/Donut, Box, Violin, and Heatmap using real genomics and transcriptomics datasets.
- **Customize**: Format typography, axis scales, tick marks, spine visibility, colorbars, legends (inside & outside axes), and annotations with arrows.
- **Assemble**: Synthesize multi-panel publication figures using `plt.subplots()` and export production-quality vector/raster graphics at 300 DPI.

## 2. Prerequisites


Before starting this notebook, you should be familiar with:
- Loading CSV files into pandas DataFrames ([`M01A`](file:///modules/M01_Python_Basics/M01A_Core_Python.ipynb)).
- Parsing FASTA sequence files with Biopython `Bio.SeqIO` ([`M02C`](file:///modules/M02_File_Handling/M02C_Biopython_SeqIO_GenBank.ipynb)).
- The Chart Selection Decision Matrix and bad plotting anti-patterns ([`M04A`](file:///modules/M04_Visualisation/M04A_Biological_Plots_Theory.ipynb)).


In [ ]:
# --- Setup -------------------------------------------------------
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install biopython seaborn requests beautifulsoup4 lxml python-dotenv
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    # Resolve repository root robustly from modules/ or repo root
    p = Path.cwd()
    while p != p.parent and not (p / "COURSE_PLAN.md").exists():
        p = p.parent
    ROOT = p if (p / "COURSE_PLAN.md").exists() else Path.cwd().resolve()

DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "M04B"
OUT_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import SeqIO
import matplotlib.ticker as ticker

print(f"Python: {sys.version.split()[0]}")
print(f"Working Directory: {Path.cwd()}")
print(f"DATA_DIR: {DATA_DIR} (exists: {DATA_DIR.exists()})")
print(f"OUT_DIR:  {OUT_DIR} (ready)")

## 4B.0 Anatomy of a Matplotlib Figure & OO Interface

### 4.0.1 Theory & Biological Context

Matplotlib is the core graphic engine of the scientific Python ecosystem. To master Matplotlib, one must understand its structural anatomy:

1. **`Figure` (`fig`)**: The top-level canvas containing all elements—one or more coordinate axes, figure titles, colorbars, and legends.
2. **`Axes` (`ax`)**: An individual plot / subplot with its own coordinate space, x-axis, y-axis, title, and data graphics. (Note: *Axes* is the singular object representing a plot, distinct from the geometrical *Axis*).
3. **`Axis` (`ax.xaxis`, `ax.yaxis`)**: The number lines establishing data limits, tick marks, and tick labels.
4. **`Spines` (`ax.spines`)**: The four enclosing bounding lines (top, bottom, left, right). Removing top and right spines is standard for publication figures.

### Two Interfaces: Pyplot vs Object-Oriented (OO)
- **Pyplot stateful interface (`plt.plot()`)**: Implicitly tracks the 'current' active figure and axes. Convenient for quick interactive scripts, but brittle for multi-panel publication figures.
- **Object-Oriented (OO) interface (`fig, ax = plt.subplots()`)**: Explicitly instantiates figure and axes objects. **This is the gold standard for bioinformatics workflows.**

### 4.0.2 Worked Example

In [ ]:
# 4B.0 Worked Example: Clean Figure & Axes Construction with OO Interface
fig, ax = plt.subplots(figsize=(7, 4), dpi=150)

# Generate biological signal simulation: e.g. GC content across a 1000 bp sliding window
x_coords = np.linspace(1, 1000, 100)
gc_signal = 50 + 12 * np.sin(x_coords / 80) + np.random.normal(0, 2, size=100)

# Plot on the Axes object
ax.plot(x_coords, gc_signal, color="#2b5c8f", lw=2, label="Window GC% (100 bp)")
ax.axhline(50, color="#d95f02", ls="--", lw=1.5, label="Genomic Average (50%)")

# Formatting Axes
ax.set_title("Anatomy of an Axes: Sliding Window GC Content", fontsize=12, pad=10, fontweight="bold")
ax.set_xlabel("Genomic Position (bp)", fontsize=10)
ax.set_ylabel("GC Percentage (%)", fontsize=10)
ax.set_ylim(25, 75)
ax.legend(frameon=True, facecolor="white", edgecolor="#cccccc", loc="upper right")

# Style spines: remove top and right for clean publication aesthetic
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", ls=":", alpha=0.5)

# Saving to OUT_DIR at 300 DPI
fig.savefig(OUT_DIR / "4B0_anatomy_demo.png", dpi=300, bbox_inches="tight")
plt.show()

### 4.0.3 ✏️ Try It: Publication Canvas Generator

Write a function `create_publication_canvas(title: str, xlabel: str, ylabel: str) -> tuple[plt.Figure, plt.Axes]`:
- Create a Figure and single Axes with `figsize=(6, 4)` and `dpi=150`.
- Set title, xlabel, and ylabel as provided.
- Hide the top and right spines (`ax.spines['top'].set_visible(False)`).
- Turn on a subtle horizontal grid line (`ax.grid(axis='y', ls=':', alpha=0.5)`).
- Return the `(fig, ax)` tuple.

In [ ]:
def create_publication_canvas(title: str, xlabel: str, ylabel: str) -> tuple[plt.Figure, plt.Axes]:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
f, a = create_publication_canvas("Gene Length", "Gene", "Length (bp)")
assert isinstance(f, plt.Figure)
assert isinstance(a, plt.Axes)
assert not a.spines["top"].get_visible()
assert not a.spines["right"].get_visible()
assert a.get_title() == "Gene Length"
plt.close(f)
print("Section 4B.0 passed ✔")

### 4.0.4 ⚠️ Common Mistakes

- **Mixing pyplot and OO syntax**: Calling `plt.title()` while working with `ax` can modify the wrong subplot if multiple subplots exist. Always use `ax.set_title()`.
- **Forgetting bbox_inches='tight'**: Without `bbox_inches='tight'` in `fig.savefig()`, rotated tick labels and legends placed outside the plot boundary are frequently clipped.

## 4B.1 Bar Plots: Vertical, Horizontal & Custom Labels

### 4.1.1 Theory & Biological Context

Bar charts display quantitative measurements across discrete categories. In biological research, common applications include:
- Gene counts per chromosome or pathway
- Differential expression fold changes across conditions
- Abundance counts of microbial taxa

### Variations:
1. **Vertical Bar (`ax.bar`)**: Default when categories have short names (e.g., Chromosomes `chr1`, `chr2`).
2. **Horizontal Bar (`ax.barh`)**: Ideal when category names are long (e.g., Gene Ontology terms, disease phenotypes).
3. **Grouped / Stacked Bar**: For comparing multiple sub-categories across main groups.
4. **Error Bars (`yerr=...`)**: Encoding biological variability (standard error or standard deviation).

### 4.1.2 Worked Example

In [ ]:
# 4B.1 Worked Example: Chromosomal Gene Counts
genes_chr_path = DATA_DIR / "tables" / "genes_per_chromosome.csv"
df_chr = pd.read_csv(genes_chr_path)

# Sort chromosomes in natural order (chr1 .. chr22, chrX, chrY) or by count
df_chr_sorted = df_chr.sort_values(by="gene_count", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 4.5), dpi=150)
bars = ax.bar(df_chr_sorted["chromosome"], df_chr_sorted["gene_count"], 
              color="#386cb0", edgecolor="#1a3b66", width=0.7)

# Add numerical values above each bar for readability
for bar in bars:
    height = bar.get_height()
    ax.annotate(f"{height:,}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),  # 3 points vertical offset
                textcoords="offset points",
                ha="center", va="bottom", fontsize=7.5, rotation=45)

ax.set_title("Human Gene Density Across Chromosomes", fontsize=12, fontweight="bold")
ax.set_xlabel("Chromosome", fontsize=10)
ax.set_ylabel("Annotated Gene Count", fontsize=10)
ax.set_ylim(0, df_chr_sorted["gene_count"].max() * 1.15)
ax.tick_params(axis="x", rotation=45)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", ls=":", alpha=0.4)

plt.tight_layout()
fig.savefig(OUT_DIR / "4B1_bar_chromosome.png", dpi=300)
plt.show()

### 4.1.3 ✏️ Try It: Horizontal Gene Count Bar Plot

Write a function `plot_horizontal_gene_bars(df: pd.DataFrame, top_n: int = 8) -> tuple[plt.Figure, plt.Axes]`:
- Take the top `top_n` chromosomes sorted by `gene_count` ascending (so largest is at top).
- Create a horizontal bar plot using `ax.barh`.
- Set title: `"Top {top_n} Chromosomes by Gene Count"`.
- Set xlabel: `"Gene Count"`, ylabel: `"Chromosome"`.
- Return `(fig, ax)`.

In [ ]:
def plot_horizontal_gene_bars(df: pd.DataFrame, top_n: int = 8) -> tuple[plt.Figure, plt.Axes]:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
f, a = plot_horizontal_gene_bars(df_chr, top_n=6)
assert len(a.patches) == 6
assert a.get_xlabel() == "Gene Count"
plt.close(f)
print("Section 4B.1 passed ✔")

### 4.1.4 ⚠️ Common Mistakes

- **Unsorted category bars**: Ordering bars randomly or alphabetically when comparing quantitative sizes forces the viewer to jump back and forth. Always sort bars by value unless there is natural domain ordering (like chr1, chr2, ...).
- **Bar plots starting above zero**: Bar lengths encode absolute quantity. Starting at a non-zero minimum distorts proportional comparison.

## 4B.2 Histograms: Bins, Density & Overlaid Distributions

### 4.2.1 Theory & Biological Context

A histogram approximates the probability density function of a continuous biological variable by binning data into adjacent non-overlapping intervals and counting occurrences.

### Key Considerations:
1. **Bin Count (`bins=...`)**: Too few bins over-smooths the distribution (masking sub-populations); too many bins produces noisy spikes. Sturges' rule ($k = \lceil \log_2 N + 1 \rceil$) or Freedman-Diaconis rule ($h = 2 \cdot \text{IQR} \cdot N^{-1/3}$) provide theoretical starting points.
2. **Density Normalization (`density=True`)**: Normalizes counts such that the total area equals 1, enabling direct comparison of datasets with unequal sample sizes.
3. **Overlaid Distributions (`alpha=...`)**: Using transparency ($0.4 - 0.6$) to compare distributions across two conditions.

### 4.2.2 Worked Example

In [ ]:
# 4B.2 Worked Example: GC Content Distribution across FASTA records
fasta_path = DATA_DIR / "sequences" / "multi_sample.fasta"

# Calculate GC content for each sequence record
gc_percentages = []
lengths = []

for record in SeqIO.parse(fasta_path, "fasta"):
    seq_str = str(record.seq).upper()
    g_count = seq_str.count("G")
    c_count = seq_str.count("C")
    gc = (g_count + c_count) / len(seq_str) * 100.0 if len(seq_str) > 0 else 0.0
    gc_percentages.append(gc)
    lengths.append(len(seq_str))

gc_percentages = np.array(gc_percentages)

fig, ax = plt.subplots(figsize=(7, 4), dpi=150)
counts, bins, patches = ax.hist(gc_percentages, bins=12, color="#74a9cf", edgecolor="#045a8d", 
                                alpha=0.8, density=True)

# Add vertical line for mean GC
mean_gc = np.mean(gc_percentages)
median_gc = np.median(gc_percentages)
ax.axvline(mean_gc, color="#d95f02", ls="--", lw=2, label=f"Mean GC: {mean_gc:.1f}%")
ax.axvline(median_gc, color="#7570b3", ls=":", lw=2, label=f"Median GC: {median_gc:.1f}%")

ax.set_title("GC% Distribution Across Biological Sequence Samples", fontsize=11, fontweight="bold")
ax.set_xlabel("GC Content (%)", fontsize=10)
ax.set_ylabel("Probability Density", fontsize=10)
ax.legend(frameon=True, loc="upper right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", ls=":", alpha=0.4)

plt.tight_layout()
fig.savefig(OUT_DIR / "4B2_hist_gc.png", dpi=300)
plt.show()

### 4.2.3 ✏️ Try It: Sequence Length Histogram

Write a function `plot_length_histogram(lengths_list: list[int], num_bins: int = 10) -> tuple[plt.Figure, plt.Axes]`:
- Plot a histogram of sequence lengths with the specified `num_bins`.
- Set bar color to `"#6baed6"` and edgecolor to `"#08519c"`.
- Set title: `"Sequence Length Distribution"`.
- Set xlabel: `"Length (bp)"`, ylabel: `"Count"`.
- Return `(fig, ax)`.

In [ ]:
def plot_length_histogram(lengths_list: list[int], num_bins: int = 10) -> tuple[plt.Figure, plt.Axes]:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
f, a = plot_length_histogram(lengths, num_bins=8)
assert len(a.patches) == 8
assert a.get_xlabel() == "Length (bp)"
plt.close(f)
print("Section 4B.2 passed ✔")

### 4.2.4 ⚠️ Common Mistakes

- **Defaulting to 10 bins without checking data scale**: Matplotlib defaults to 10 bins, which may obscure multi-modality in large genomic datasets. Always experiment with bin counts.
- **Comparing raw counts across groups of different sample sizes**: Always use `density=True` when overlaying histograms of groups with unequal N.

## 4B.3 Pie & Donut Charts: Nucleotide Composition

### 4.3.1 Theory & Biological Context

Pie charts divide a circular graphic into proportional slices. While frequently overused in business presentations, pie and donut charts can be justified in biological contexts under strict constraints:
- **Maximum 4–5 categories**.
- **Data represents true parts of a whole (must sum to 100%)**.
- Classic biological example: Nucleotide base composition of a transcript (Adenine, Cytosine, Guanine, Thymine/Uracil).

### Donut Charts:
A donut chart includes a hollow center (created via `plt.Circle` or `wedgeprops=dict(width=0.4)`). This improves aesthetic appeal and allows placing key summary metrics (e.g. total sequence length) in the center.

### 4.3.2 Worked Example

In [ ]:
# 4B.3 Worked Example: Base Composition of Human TP53 mRNA
tp53_fasta = DATA_DIR / "sequences" / "tp53_human_mrna.fasta"
tp53_record = SeqIO.read(tp53_fasta, "fasta")
seq_clean = str(tp53_record.seq).upper()

base_counts = {
    "Adenine (A)": seq_clean.count("A"),
    "Cytosine (C)": seq_clean.count("C"),
    "Guanine (G)": seq_clean.count("G"),
    "Thymine (T)": seq_clean.count("T")
}

labels = list(base_counts.keys())
values = list(base_counts.values())
colors = ["#66c2a5", "#fc8d62", "#8da0cb", "#e78ac3"]

fig, ax = plt.subplots(figsize=(6, 6), dpi=150)
wedges, texts, autotexts = ax.pie(
    values, labels=labels, autopct="%1.1f%%",
    startangle=140, colors=colors,
    pctdistance=0.75,
    wedgeprops=dict(width=0.4, edgecolor="white", lw=2)  # width creates the donut hole!
)

# Style text inside donut
for autotext in autotexts:
    autotext.set_color("black")
    autotext.set_fontweight("bold")

# Add summary text in the central donut hole
ax.text(0, 0, f"TP53 mRNA\n{len(seq_clean):,} bp", ha="center", va="center", 
        fontsize=11, fontweight="bold", color="#333333")

ax.set_title("Nucleotide Composition: Human TP53 mRNA", fontsize=12, fontweight="bold", pad=15)
plt.tight_layout()
fig.savefig(OUT_DIR / "4B3_donut_tp53.png", dpi=300)
plt.show()

### 4.3.3 ✏️ Try It: Exploded Nucleotide Pie Chart

Write a function `plot_exploded_base_pie(base_dict: dict) -> tuple[plt.Figure, plt.Axes]`:
- Takes a dictionary of nucleotide counts `{'A': ..., 'C': ..., 'G': ..., 'T': ...}`.
- Create an exploded pie chart where the slice for `'G'` is exploded by `0.1` (`explode=[0, 0, 0.1, 0]`).
- Set colors: `['#4daf4a', '#377eb8', '#ff7f00', '#e41a1c']`.
- Set `autopct='%1.1f%%'`.
- Return `(fig, ax)`.

In [ ]:
def plot_exploded_base_pie(base_dict: dict) -> tuple[plt.Figure, plt.Axes]:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
f, a = plot_exploded_base_pie({'A': 300, 'C': 250, 'G': 350, 'T': 200})
assert len(a.patches) == 4
plt.close(f)
print("Section 4B.3 passed ✔")

### 4.3.4 ⚠️ Common Mistakes

- **Using pie charts for non-proportional data**: Never use pie charts for data that does not sum to 100% of a defined whole.
- **Using 3D pie charts**: 3D perspective distorts slice angles and causes deceptive perception of proportions.

## 4B.4 Box Plots: Quartiles, Notches & Outliers

### 4.4.1 Theory & Biological Context

A box plot (Tukey box-and-whisker plot) provides a compact, non-parametric summary of a distribution:
- **Central line**: Median ($Q_2$, 50th percentile).
- **Box boundaries**: First quartile ($Q_1$, 25th percentile) and Third quartile ($Q_3$, 75th percentile).
- **Interquartile Range (IQR)**: $IQR = Q_3 - Q_1$, enclosing 50% of the sample mass.
- **Whiskers**: Extend to the most extreme data points within $1.5 \times IQR$ from the box hinges.
- **Fliers (Outliers)**: Points falling beyond the whiskers ($> 1.5 \times IQR$).

### Notched Box Plots:
Notches represent the $95\%$ confidence interval around the median:
$$\text{Notch} \approx \text{Median} \pm 1.57 \times \frac{IQR}{\sqrt{N}}$$
If the notches of two boxes do not overlap, their medians differ significantly at the $5\%$ level.

### 4.4.2 Worked Example

In [ ]:
# 4B.4 Worked Example: Gene Expression Across Tissues
tissues_path = DATA_DIR / "tables" / "expression_tissues.csv"
df_tissues = pd.read_csv(tissues_path)

# Extract TPM distributions per tissue
tissue_names = df_tissues["tissue"].unique()
tissue_data = [df_tissues[df_tissues["tissue"] == t]["tpm"].values for t in tissue_names]

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)
bp = ax.boxplot(tissue_data, patch_artist=True, notch=True,
                boxprops=dict(facecolor="#9ecae1", color="#08519c", lw=1.5),
                medianprops=dict(color="#d95f02", lw=2),
                whiskerprops=dict(color="#08519c", lw=1.2),
                capprops=dict(color="#08519c", lw=1.2),
                flierprops=dict(marker="o", color="#de2d26", alpha=0.6, markersize=5))

ax.set_xticklabels(tissue_names, fontsize=10)
ax.set_title("Tissue-Specific Expression Profile (TPM)", fontsize=12, fontweight="bold")
ax.set_xlabel("Tissue", fontsize=10)
ax.set_ylabel("Expression Level (TPM)", fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", ls=":", alpha=0.4)

plt.tight_layout()
fig.savefig(OUT_DIR / "4B4_boxplot_tissues.png", dpi=300)
plt.show()

### 4.4.3 ✏️ Try It: Custom Styled Box Plot

Write a function `generate_custom_boxplot(data_arrays: list[np.ndarray], labels: list[str]) -> tuple[plt.Figure, plt.Axes]`:
- Create a box plot with `patch_artist=True`.
- Color each box with `facecolor="#c7e9c0"` and median color `"#006d2c"`.
- Set xticklabels to `labels`.
- Set ylabel to `"Measurement (AU)"`.
- Return `(fig, ax)`.

In [ ]:
def generate_custom_boxplot(data_arrays: list[np.ndarray], labels: list[str]) -> tuple[plt.Figure, plt.Axes]:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
f, a = generate_custom_boxplot([np.array([1, 2, 3, 4, 5]), np.array([3, 4, 5, 6, 7])], ["A", "B"])
assert len(a.get_xticklabels()) == 2
plt.close(f)
print("Section 4B.4 passed ✔")

### 4.4.4 ⚠️ Common Mistakes

- **Omitting patch_artist=True**: Calling `ax.boxplot()` without `patch_artist=True` ignores `facecolor` because boxes remain un-filled line artists.
- **Assuming box plots prove normality**: Box plots are non-parametric; they do not indicate whether data follows a Gaussian distribution.

## 4B.5 Violin Plots: Distribution Shapes & Multimodality

### 4.5.1 Theory & Biological Context

A violin plot combines a box plot with a rotated Kernel Density Estimation (KDE) curve on each side.

### Advantages Over Box Plots:
- **Reveals Multimodal Distributions**: Box plots cannot distinguish a unimodal distribution from a bimodal distribution with the same median and IQR. Violin plots clearly reveal twin peaks (subpopulations).
- Particularly vital in single-cell transcriptomics (e.g. distinguishing non-expressing cells from expressing subpopulations).

### Matplotlib `ax.violinplot` Return Structure:
Unlike `ax.boxplot`, `ax.violinplot` returns a dictionary of Artists:
- `'bodies'`: A list of `PolyCollection` objects representing the violin bodies.
- `'cbars'`, `'cmaxes'`, `'cmins'`, `'cmedians'`: `LineCollection` objects for the bounding lines.

### 4.5.2 Worked Example

In [ ]:
# 4B.5 Worked Example: Violin Plot of Tissue Expression
fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)

# Generate violin plot
vp = ax.violinplot(tissue_data, showmedians=True, showextrema=True)

# Customise violin bodies
colors = ["#7293CB", "#E1974C", "#84BA5B", "#D35E60", "#808585"]
for idx, body in enumerate(vp["bodies"]):
    body.set_facecolor(colors[idx % len(colors)])
    body.set_edgecolor("#333333")
    body.set_alpha(0.7)

# Customise median lines
vp["cmedians"].set_color("black")
vp["cmedians"].set_linewidth(2)

ax.set_xticks(np.arange(1, len(tissue_names) + 1))
ax.set_xticklabels(tissue_names, fontsize=10)
ax.set_title("Expression Distribution Across Tissues (Violin Plot)", fontsize=12, fontweight="bold")
ax.set_ylabel("TPM Expression", fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", ls=":", alpha=0.4)

plt.tight_layout()
fig.savefig(OUT_DIR / "4B5_violin_tissues.png", dpi=300)
plt.show()

### 4.5.3 ✏️ Try It: Styled Violin Plot

Write a function `plot_styled_violins(data: list[np.ndarray], labels: list[str]) -> tuple[plt.Figure, plt.Axes]`:
- Plot violins using `ax.violinplot(..., showmedians=True)`.
- Set all violin bodies to `facecolor="#bcbddc"`, `edgecolor="#756bb1"`, `alpha=0.6`.
- Set x-ticks and labels corresponding to `labels`.
- Return `(fig, ax)`.

In [ ]:
def plot_styled_violins(data: list[np.ndarray], labels: list[str]) -> tuple[plt.Figure, plt.Axes]:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
f, a = plot_styled_violins([np.random.normal(size=50), np.random.normal(2, 1, size=50)], ["Ctrl", "Treat"])
assert len(a.get_xticks()) == 2
plt.close(f)
print("Section 4B.5 passed ✔")

### 4.5.4 ⚠️ Common Mistakes

- **Passing a 2D DataFrame directly to ax.violinplot**: Matplotlib's `ax.violinplot` does not parse DataFrame column names automatically; pass a list of 1D arrays.
- **Forgetting to set xticks manually**: Matplotlib positions violins at integer coordinates `1, 2, ..., N`. You must set `ax.set_xticks(np.arange(1, N + 1))` before setting `xticklabels`.

## 4B.6 Heatmaps with imshow: Matrix Visualization & Colorbars

### 4.6.1 Theory & Biological Context

A biological heatmap displays a 2D matrix of continuous values as colored cells.
Primary use cases:
- Gene expression matrices (Genes $\times$ Biological Samples)
- Correlation matrices (Pairwise Pearson $r$ between biomarkers)
- Mutation burden or variant matrices

### Key Matplotlib Functions:
- `ax.imshow(matrix, cmap=..., aspect='auto')`: Renders 2D array.
- `fig.colorbar(im, ax=ax)`: Appends color scale bar.
- `ax.set_xticks` / `ax.set_yticks`: Aligns tick marks to cell centers.
- Cell annotation: Iterating with `ax.text(j, i, f'{val:.2f}')` to print values inside cells.

### 4.6.2 Worked Example

In [ ]:
# 4B.6 Worked Example: Gene Expression Heatmap with Value Annotations
exp_mat_path = DATA_DIR / "tables" / "expression_matrix.csv"
df_mat = pd.read_csv(exp_mat_path)

# Select first 6 genes and the 8 sample columns
sample_cols = [c for c in df_mat.columns if c != "gene"]
df_sub = df_mat.head(6)
gene_names = df_sub["gene"].values
mat_vals = df_sub[sample_cols].values

# Log2 transform for balanced dynamic range: log2(x + 1)
log_vals = np.log2(mat_vals + 1.0)

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)
im = ax.imshow(log_vals, cmap="viridis", aspect="auto")

# Colorbar
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Expression Level: log2(TPM + 1)", fontsize=9)

# Align ticks to grid cell coordinates
ax.set_xticks(np.arange(len(sample_cols)))
ax.set_yticks(np.arange(len(gene_names)))
ax.set_xticklabels(sample_cols, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(gene_names, fontsize=9, fontweight="bold")

# Annotate cell numbers
for i in range(len(gene_names)):
    for j in range(len(sample_cols)):
        val = log_vals[i, j]
        # Use white text for dark cells, black text for bright cells
        text_color = "white" if val < np.mean(log_vals) else "black"
        ax.text(j, i, f"{val:.1f}", ha="center", va="center", color=text_color, fontsize=7.5)

ax.set_title("Target Gene Expression Heatmap Across Samples", fontsize=11, fontweight="bold", pad=10)
plt.tight_layout()
fig.savefig(OUT_DIR / "4B6_heatmap_matrix.png", dpi=300)
plt.show()

### 4.6.3 ✏️ Try It: Correlation Heatmap

Write a function `plot_correlation_heatmap(corr_matrix: np.ndarray, labels: list[str]) -> tuple[plt.Figure, plt.Axes]`:
- Render heatmap using `ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1.0, vmax=1.0)`.
- Set x and y tick labels to `labels`.
- Add a colorbar with label `"Pearson r"`.
- Return `(fig, ax)`.

In [ ]:
def plot_correlation_heatmap(corr_matrix: np.ndarray, labels: list[str]) -> tuple[plt.Figure, plt.Axes]:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
dummy_corr = np.array([[1.0, 0.8], [0.8, 1.0]])
f, a = plot_correlation_heatmap(dummy_corr, ["G1", "G2"])
assert len(a.get_xticklabels()) == 2
plt.close(f)
print("Section 4B.6 passed ✔")

### 4.6.4 ⚠️ Common Mistakes

- **Asymmetric diverging colormap ranges**: If plotting correlation from -0.3 to 1.0 without setting `vmin=-1, vmax=1`, zero will not align with neutral white, severely misleading visual interpretation.
- **Misaligned tick marks**: Failing to pass `np.arange(len(labels))` causes ticks to misalign with cell boundaries.

## 4B.7 Customisation & Multi-Panel Publication Figures

### 4.7.1 Theory & Biological Context

Publication figures in Nature, Cell, or Science are rarely single isolated plots. They are composite multi-panel figures (Panel A, B, C, D) assembled with harmonious typography, coordinated color palettes, legends positioned outside data areas, and arrow annotations pointing out key biological discoveries.

### Key Tools:
1. **Multi-panel layouts**: `fig, axes = plt.subplots(nrows, ncols, figsize=(w, h))`.
2. **Legends Outside Plot**: `ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')`.
3. **Annotations**: `ax.annotate('Key finding', xy=(x, y), xytext=(xt, yt), arrowprops=dict(arrowstyle='->'))`.
4. **Figure-level Labels**: `ax.text(-0.1, 1.05, 'A', transform=ax.transAxes, fontsize=14, fontweight='bold')`.

### 4.7.2 Worked Example

In [ ]:
# 4B.7 Worked Example: Multi-Panel Publication Composite Figure
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), dpi=150)

# Panel A: Chromosome bar plot
top_5_chr = df_chr.nlargest(5, "gene_count")
axes[0].bar(top_5_chr["chromosome"], top_5_chr["gene_count"], color="#41b6c4", edgecolor="#225ea8")
axes[0].set_title("Top 5 Gene-Dense Chromosomes", fontsize=10, fontweight="bold")
axes[0].set_xlabel("Chromosome")
axes[0].set_ylabel("Gene Count")
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)
axes[0].text(-0.15, 1.05, "A", transform=axes[0].transAxes, fontsize=14, fontweight="bold")

# Annotate highest chromosome
top_val = top_5_chr["gene_count"].iloc[0]
axes[0].annotate(f"Highest ({top_val:,})", xy=(0, top_val), xytext=(0.5, top_val * 0.9),
                 arrowprops=dict(arrowstyle="->", color="darkred", lw=1.5),
                 fontweight="bold", color="darkred", fontsize=8.5)

# Panel B: Box plot of Tissue Expression
axes[1].boxplot(tissue_data[:4], patch_artist=True,
                boxprops=dict(facecolor="#e7298a", alpha=0.4),
                medianprops=dict(color="#980043", lw=2))
axes[1].set_xticklabels(tissue_names[:4], fontsize=9)
axes[1].set_title("Expression Across 4 Vital Tissues", fontsize=10, fontweight="bold")
axes[1].set_ylabel("TPM Expression")
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)
axes[1].text(-0.15, 1.05, "B", transform=axes[1].transAxes, fontsize=14, fontweight="bold")

plt.tight_layout()
fig.savefig(OUT_DIR / "4B7_composite_figure.png", dpi=300, bbox_inches="tight")
plt.show()

### 4.7.3 ✏️ Try It: Dual-Panel Assembler

Write a function `assemble_dual_panel(title_a: str, title_b: str) -> tuple[plt.Figure, np.ndarray]`:
- Create a 1x2 subplots canvas with `figsize=(10, 4)`.
- Set titles on `axes[0]` and `axes[1]`.
- Add panel letters `'A'` and `'B'` at `transform=ax.transAxes` at `(-0.12, 1.05)`.
- Return `(fig, axes)`.

In [ ]:
def assemble_dual_panel(title_a: str, title_b: str) -> tuple[plt.Figure, np.ndarray]:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
f, axs = assemble_dual_panel("Cohort 1", "Cohort 2")
assert len(axs) == 2
plt.close(f)
print("Section 4B.7 passed ✔")

### 4.7.4 ⚠️ Common Mistakes

- **Hardcoding coordinates in data space for panel labels**: Using data coordinates for 'A' and 'B' labels causes them to jump around when data scales change. Always use `transform=ax.transAxes`.
- **Crowded unaligned subplots**: Forgetting `plt.tight_layout()` or `constrained_layout=True` causes overlapping tick labels and titles in multi-panel grids.

### Practice Exercise 4B.1 (★) — Top 10 Chromosomes Horizontal Bar Chart

**Problem**: Write a function `save_top_chromosomes_barchart(csv_path: Path, out_path: Path) -> Path`:
- Read `genes_per_chromosome.csv`.
- Select the top 10 chromosomes with the highest `gene_count`, sorted ascending (so the largest bar is at the top).
- Create a horizontal bar chart with bar color `"#2b8cbe"` and edge color `"#045a8d"`.
- Set title: `"Top 10 Gene-Dense Human Chromosomes"`.
- Remove top and right spines, add horizontal/vertical dotted grid.
- Save to `out_path` with `dpi=300, bbox_inches='tight'`.
- Return `out_path`.


- **Input**: Path to genes_per_chromosome.csv and output destination Path
- **Expected Output**: Saved figure file path

In [ ]:
def save_top_chromosomes_barchart(csv_path: Path, out_path: Path) -> Path:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
out_file = OUT_DIR / "top10_chromosomes.png"
p = save_top_chromosomes_barchart(DATA_DIR / "tables" / "genes_per_chromosome.csv", out_file)
assert p.exists() and p.stat().st_size > 15000
print("Ex 4B.1 passed ✔")

### Practice Exercise 4B.2 (★★) — Comparative Overlaid GC% and AT% Histogram

**Problem**: Write a function `plot_comparative_base_hist(fasta_path: Path, out_path: Path) -> Path`:
- Parse sequences from `multi_sample.fasta`.
- Calculate GC% and AT% for each sequence record.
- Create an overlaid histogram with 10 bins:
  - GC%: color `"#3182bd"`, `alpha=0.6`, label `"GC Content"`.
  - AT%: color `"#de2d26"`, `alpha=0.6`, label `"AT Content"`.
- Add vertical dashed line for mean GC and mean AT.
- Include a formatted legend in the upper right.
- Set title: `"Comparative Nucleotide Distribution"`, xlabel: `"Percentage (%)"`, ylabel: `"Frequency"`.
- Save figure to `out_path` with `dpi=300, bbox_inches='tight'`.
- Return `out_path`.


- **Input**: Path to multi_sample.fasta and target image Path
- **Expected Output**: Path to saved figure

In [ ]:
def plot_comparative_base_hist(fasta_path: Path, out_path: Path) -> Path:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
out_hist = OUT_DIR / "overlaid_base_hist.png"
p_hist = plot_comparative_base_hist(DATA_DIR / "sequences" / "multi_sample.fasta", out_hist)
assert p_hist.exists() and p_hist.stat().st_size > 15000
print("Ex 4B.2 passed ✔")

### Practice Exercise 4B.3 (★★★) — Composite Publication Figure: Expression Boxplot + Correlation Heatmap

**Problem**: Write a function `generate_publication_figure(tissue_csv: Path, matrix_csv: Path, out_path: Path) -> Path`:
- Construct a 1x2 multi-panel figure with `figsize=(12, 5)` and `dpi=300`.
- **Panel A (Left)**: Box plot of TPM across all tissues from `expression_tissues.csv`.
  - Style with `patch_artist=True`, custom colors, median lines.
  - Add panel tag `'A'` at `(-0.1, 1.05)`.
- **Panel B (Right)**: Heatmap of Pearson correlation matrix between the top 6 genes from `expression_matrix.csv`.
  - Use diverging `cmap='vlag'`, `vmin=-1.0, vmax=1.0`.
  - Annotate each cell with the correlation rounded to 2 decimals.
  - Add colorbar with label `'Pearson r'`.
  - Add panel tag `'B'` at `(-0.1, 1.05)`.
- Apply publication styling (clean spines, font sizes, tight layout).
- Save to `out_path` and return it.


- **Input**: Paths to tissue CSV, matrix CSV, and output PNG Path
- **Expected Output**: Path to saved publication composite figure

In [ ]:
def generate_publication_figure(tissue_csv: Path, matrix_csv: Path, out_path: Path) -> Path:
    # TODO: your code here
    ...

In [ ]:
# ✅ Test — run me. No output means you passed.
out_composite = OUT_DIR / "publication_composite.png"
p_comp = generate_publication_figure(DATA_DIR / "tables" / "expression_tissues.csv",
                                     DATA_DIR / "tables" / "expression_matrix.csv",
                                     out_composite)
assert p_comp.exists() and p_comp.stat().st_size > 25000
print("Ex 4B.3 passed ✔")

## 6. Summary & Cheat Sheet

Matplotlib provides full pixel-level control over every visual element. Adopting the Object-Oriented interface (`fig, ax = plt.subplots()`) enables clean code architecture for multi-panel composite figures, custom annotations, and publication-ready export.

### Quick Reference Table

| Concept / Operation | Syntax / Function | Biological Application | Key Parameter / Note |
| --- | --- | --- | --- |
| Subplot Creation | fig, ax = plt.subplots(nrows, ncols, figsize) | Canvas instantiation | Always use OO syntax |
| Bar Plot | ax.bar(x, y) / ax.barh(y, x) | Categorical counts | y-axis MUST start at 0 |
| Histogram | ax.hist(data, bins=..., density=True) | Continuous distributions | Use density=True for comparative N |
| Donut Chart | ax.pie(vals, wedgeprops=dict(width=0.4)) | Parts of whole (max 5 cats) | Add central summary label |
| Box Plot | ax.boxplot(data, patch_artist=True) | Median, IQR, outliers | Requires patch_artist for colors |
| Violin Plot | ax.violinplot(data, showmedians=True) | Multimodal distribution shapes | Style bodies via vp['bodies'] |
| Heatmap | ax.imshow(mat, cmap=...) + fig.colorbar() | Matrices and correlations | Align ticks with np.arange |
| Export | fig.savefig(path, dpi=300, bbox_inches='tight') | Publication submission | Always specify tight bbox |

## 7. Further Reading & Resources

- [Matplotlib Official Documentation & Gallery](https://matplotlib.org/stable/gallery/index.html): Comprehensive visual example index.
- [Scientific Visualization: Python + Matplotlib](https://hal.inria.fr/hal-03427242): Nicolas Rougier's open-access masterwork on high-aesthetic scientific plotting.
- [Nature Methods: Points of Significance (Plotting Distributions)](https://www.nature.com/articles/nmeth.2813): Krzywinski & Altman on box plots and violin plots.